# Session 2.2 : Tidy Data

_Analytics Through Coding Autumn 2026_

---

Data are not always stored in the structure that makes analysis easy.

The purpose of **tidy data** is to give us a consistent way of organising a dataset so that pandas operations such as filtering, grouping, aggregation and visualisation work naturally.

In this session we will focus on two common reshaping problems:

* a variable is spread across several columns;
* an observation is spread across several rows.

We will use:

* `melt()` to reshape data from **wide to long**;
* `pivot()` / `pivot_table()` to reshape data from **long to wide**.

---

## Starting out

As always, import the libraries we need at the beginning of the notebook.

In [1]:
import pandas as pd
import numpy as np

## What does tidy data mean?

There are three interrelated rules:

**1. Each variable has its own column.**

**2. Each observation has its own row.**

**3. Each value has its own cell.**

The important question is always:

> **What does one row represent?**

If we cannot answer that clearly, the structure of the dataset may need to change.

![GitHub Codespaces](tidy_data.png)


## The same information can be stored in different ways

The small examples below contain the same underlying information about tuberculosis cases and population, but they are organised differently.

In [2]:
table1 = pd.DataFrame({
    "country": ["Afghanistan", "Afghanistan", "Brazil", "Brazil", "China", "China"],
    "year": [1999, 2000, 1999, 2000, 1999, 2000],
    "cases": [745, 2666, 37737, 80488, 212258, 213766],
    "population": [19987071, 20595360, 172006362, 174504898, 1272915272, 1280428583]
})

table2 = pd.DataFrame({
    "country": ["Afghanistan", "Afghanistan", "Afghanistan", "Afghanistan",
                "Brazil", "Brazil", "Brazil", "Brazil",
                "China", "China", "China", "China"],
    "year": [1999, 1999, 2000, 2000,
             1999, 1999, 2000, 2000,
             1999, 1999, 2000, 2000],
    "type": ["cases", "population"] * 6,
    "count": [745, 19987071, 2666, 20595360,
              37737, 172006362, 80488, 174504898,
              212258, 1272915272, 213766, 1280428583]
})

table4a = pd.DataFrame({
    "country": ["Afghanistan", "Brazil", "China"],
    "1999": [745, 37737, 212258],
    "2000": [2666, 80488, 213766]
})

table4b = pd.DataFrame({
    "country": ["Afghanistan", "Brazil", "China"],
    "1999": [19987071, 172006362, 1272915272],
    "2000": [20595360, 174504898, 1280428583]
})

display(table1)
display(table2)
display(table4a)

,country,year,cases,population
0,Afghanistan,1999,745,19987071
1,Afghanistan,2000,2666,20595360
2,Brazil,1999,37737,172006362
3,Brazil,2000,80488,174504898
4,China,1999,212258,1272915272
5,China,2000,213766,1280428583


,country,year,type,count
0,Afghanistan,1999,cases,745
1,Afghanistan,1999,population,19987071
2,Afghanistan,2000,cases,2666
3,Afghanistan,2000,population,20595360
4,Brazil,1999,cases,37737
5,Brazil,1999,population,172006362
6,Brazil,2000,cases,80488
7,Brazil,2000,population,174504898
8,China,1999,cases,212258
9,China,1999,population,1272915272


,country,1999,2000
0,Afghanistan,745,2666
1,Brazil,37737,80488
2,China,212258,213766


`table1` is tidy because:

* `country`, `year`, `cases` and `population` are variables;
* each country-year combination is one observation;
* each value occupies one cell.

The other tables are not necessarily *wrong*. They are simply less convenient for some kinds of analysis.

The structure we choose depends on what we need to do next.

## Wide to long: `melt()`

In `table4a`, the years `1999` and `2000` appear as **column names**.

But year is really a variable.

To make the data longer, we use `melt()`.

In [4]:
table4a_long = pd.melt(table4a, id_vars=["country"], var_name="year", value_name="cases")
table4a_long.head()

,country,year,cases
0,Afghanistan,1999,745
1,Brazil,1999,37737
2,China,1999,212258
3,Afghanistan,2000,2666
4,Brazil,2000,80488


The arguments tell pandas:

* `id_vars` — columns that stay fixed;
* `var_name` — the name of the new variable created from the old column names;
* `value_name` — the name of the column containing the values.

After reshaping:

> one row = one country-year observation.

### Exercise 1 — Reshape population data

`table4b` contains population values for 1999 and 2000 in separate columns.

Use `.melt()` to reshape it so that the resulting DataFrame has:

* `country`
* `year`
* `population`

Call the new DataFrame `table4b_long`.

In [3]:
table4b_long = pd.melt(table4b, id_vars=["country"], var_name="year", value_name="population")
table4b_long.head()

,country,year,population
0,Afghanistan,1999,19987071
1,Brazil,1999,172006362
2,China,1999,1272915272
3,Afghanistan,2000,20595360
4,Brazil,2000,174504898


## Long to wide: `pivot()`

Sometimes we have the opposite problem.

In `table2`, cases and population are stored in separate **rows** for the same country-year observation.

In [6]:
table2_wide = table2.pivot_table(index=["country", "year"], columns="type", values="count").reset_index()

table2_wide.head()

type,country,year,cases,population
0,Afghanistan,1999,745.0,1.998707e+07
1,Afghanistan,2000,2666.0,2.059536e+07
2,Brazil,1999,37737.0,1.720064e+08
3,Brazil,2000,80488.0,1.745049e+08
4,China,1999,212258.0,1.272915e+09


If every combination of `country`, `year` and `type` is unique, we can use `.pivot()`.

The resulting data are now in the same basic structure as `table1`.

Here:

* `index` identifies each observation;
* `columns` tells pandas which values should become column names;
* `values` tells pandas which values should fill those cells.

### Exercise 2 — Return to a tidy structure

Use `.pivot()` on `table2` so that:

* one row represents one country-year observation;
* `cases` and `population` become separate columns.

Store the result in `table2_tidy`.

In [8]:
table2_tidy = table2.pivot_table(index=["country", "year"], columns="type", values="count").reset_index()

table2_wide.head()

type,country,year,cases,population
0,Afghanistan,1999,745.0,1.998707e+07
1,Afghanistan,2000,2666.0,2.059536e+07
2,Brazil,1999,37737.0,1.720064e+08
3,Brazil,2000,80488.0,1.745049e+08
4,China,1999,212258.0,1.272915e+09


## `pivot()` versus `pivot_table()`

`.pivot()` requires the combination of the index and column variables to be **unique**.

If duplicate combinations exist, pandas cannot know which value should go into the cell.

When duplicates need to be combined, use `.pivot_table()` and specify an aggregation function.

In [10]:
example = pd.DataFrame({
    "country": ["A", "A", "A", "A"],
    "year": [2025, 2025, 2025, 2025],
    "type": ["cases", "cases", "population", "population"],
    "count": [10, 5, 1000, 2000]
})
example

,country,year,type,count
0,A,2025,cases,10
1,A,2025,cases,5
2,A,2025,population,1000
3,A,2025,population,2000


In [12]:
example_wide=example.pivot_table(
    index=["country", "year"], columns="type", values="count", aggfunc="sum").reset_index()

example_wide

type,country,year,cases,population
0,A,2025,15,3000


<div class="alert alert-warning">
<b>Important.</b>
Do not use <code>pivot_table()</code> simply because <code>pivot()</code> fails.

A failed pivot may be telling you that duplicate observations exist. First ask whether those duplicates are expected and what they represent.
</div>

## Real-world example: WHO tuberculosis data

Small examples make the reshaping operation easy to see, but real datasets can be much messier.

The WHO dataset records tuberculosis cases by country and year.

Many combinations of case type, sex and age group are stored as separate columns, for example:

* `new_sp_m014`
* `new_sp_f1524`

This makes the dataset very wide.

Our goal here is **not** to clean every part of the WHO dataset. The goal is to recognise the structural problem and reshape it into a form that is easier to analyse.

In [13]:
who = pd.read_csv("../Data/who.csv")

print("Shape:", who.shape)
who.head()

Shape: (7240, 60)


,country,iso2,iso3,year,new_sp_m014,new_sp_m1524,new_sp_m2534,new_sp_m3544,new_sp_m4554,new_sp_m5564,...,newrel_m4554,newrel_m5564,newrel_m65,newrel_f014,newrel_f1524,newrel_f2534,newrel_f3544,newrel_f4554,newrel_f5564,newrel_f65
0,Afghanistan,AF,AFG,1980,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AF,AFG,1981,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AF,AFG,1982,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AF,AFG,1983,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AF,AFG,1984,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Look at the column names.

What do you notice?

Many of the column names are not really variable names — they contain **values describing categories**.

In [14]:
who.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['country', 'iso2', 'iso3', 'year', 'new_sp_m014', 'new_sp_m1524',
       'new_sp_m2534', 'new_sp_m3544', 'new_sp_m4554', 'new_sp_m5564',
       'new_sp_m65', 'new_sp_f014', 'new_sp_f1524', 'new_sp_f2534',
       'new_sp_f3544', 'new_sp_f4554', 'new_sp_f5564', 'new_sp_f65',
       'new_sn_m014', 'new_sn_m1524', 'new_sn_m2534', 'new_sn_m3544',
       'new_sn_m4554', 'new_sn_m5564', 'new_sn_m65', 'new_sn_f014',
       'new_sn_f1524', 'new_sn_f2534', 'new_sn_f3544', 'new_sn_f4554',
       'new_sn_f5564', 'new_sn_f65', 'new_ep_m014', 'new_ep_m1524',
       'new_ep_m2534', 'new_ep_m3544', 'new_ep_m4554', 'new_ep_m5564',
       'new_ep_m65', 'new_ep_f014', 'new_ep_f1524', 'new_ep_f2534',
       'new_ep_f3544', 'new_ep_f4554', 'new_ep_f5564', 'new_ep_f65',
       'newrel_m014', 'newrel_m1524', 'newrel_m2534', 'newrel_m3544',
       'newrel_m4554', 'newrel_m5564', 'newrel_m65', 'newrel_f014',
       'newrel_f1524', 'newrel_f2534', 'newrel_f3544', 'ne

### Final Exercise — Reshape WHO tuberculosis data

Create a longer version of the WHO dataset.

1. Use `pd.melt()` to convert columns 5 through 60 into:
   * `tb_codes` — the original column name;
   * `n` — the number of TB cases.
2. Keep `country` and `year` as identifier variables.
3. Remove observations where `n` is missing.
4. Compare the shape before and after removing missing values.
5. Display the first 10 rows of the resulting data.

Call the final DataFrame `who_long`.

In [ ]:
who_long = pd.melt(
    who, id_vars=["country", "year"], 
    value_vars=who.columns[4:60], var_name="tb_code", value_name ='n').query("n.notnull()").reset_index(drop=True)

#who_long.dropna(subset=["n"], inplace=True)

In [35]:
who.head()

,country,iso2,iso3,year,new_sp_m014,new_sp_m1524,new_sp_m2534,new_sp_m3544,new_sp_m4554,new_sp_m5564,...,newrel_m4554,newrel_m5564,newrel_m65,newrel_f014,newrel_f1524,newrel_f2534,newrel_f3544,newrel_f4554,newrel_f5564,newrel_f65
0,Afghanistan,AF,AFG,1980,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AF,AFG,1981,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AF,AFG,1982,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AF,AFG,1983,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AF,AFG,1984,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
who_long.head()

,country,year,tb_code,n
0,Afghanistan,1997,new_sp_m014,0.0
1,Afghanistan,1998,new_sp_m014,30.0
2,Afghanistan,1999,new_sp_m014,8.0
3,Afghanistan,2000,new_sp_m014,52.0
4,Afghanistan,2001,new_sp_m014,129.0


## Check the reshaped data

Reshaping is a transformation, so we should check the result rather than assuming it worked simply because the code ran.

Ask:

* What does one row represent now?
* Did the number of rows change in the way we expected?
* Are the identifier variables still present?
* Are the values in the new columns plausible?
* Did we lose information unintentionally?

In [39]:
who_long['country'].nunique()

who_long.describe()

,year,n
count,76046.000000,76046.000000
mean,2006.165742,570.674565
std,5.147049,3544.814953
min,1980.000000,0.000000
25%,2003.000000,3.000000
50%,2007.000000,26.000000
75%,2010.000000,184.000000
max,2013.000000,250051.000000


## Optional extension — Baby names

If time permits, the UK baby names dataset provides another useful reshaping example.

The data are stored in long form with variables including:

* `year`
* `sex`
* `name`
* `n`

Suppose we want one row per year and separate columns for male and female counts.

Because there are many names within each year-sex combination, we need `pivot_table()` rather than `pivot()`.

In [41]:
# Optional

babynames = pd.read_csv("../Data/ukbabynames.csv")



In [42]:
babynames.head()

,year,sex,name,n,rank,nation
0,1996,F,SOPHIE,7087,1,England & Wales
1,1996,F,CHLOE,6824,2,England & Wales
2,1996,F,JESSICA,6711,3,England & Wales
3,1996,F,EMILY,6415,4,England & Wales
4,1996,F,LAUREN,6299,5,England & Wales


### Optional challenge

Using `babies_by_sex`, calculate the percentage of babies who are female in each year.

## All Done!

In this session we focused on the **structure of data**, rather than learning many new pandas commands.

We practised:

* recognising tidy and untidy structures;
* identifying what one row represents;
* reshaping wide data to long data with `melt()`;
* reshaping long data to wide data with `pivot()`;
* using `pivot_table()` when multiple observations need to be aggregated;
* checking that reshaping produced the structure we expected.

The key idea is:

> **Data should be organised in a structure that makes the analytical question easier to answer.**

We will now move on to **relational data**, where the challenge is not reshaping one table but combining information stored across multiple tables.